# Visualisations approfondies — CORTI 500

Analyse démographique, distribution PTA, asymétrie, corrélations ML vs clinique

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.loader import load_json_file
from src.features import build_feature_matrix, preprocess, VISIT_CATEGORY_LABELS, STANDARD_FREQS, NIHL_NOTCH_THRESHOLD_DB, compute_nihl_flag
from src.models.unsupervised import run_unsupervised_pipeline
from src.evaluate import plot_audiogram, plot_patient_trajectory

plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

GENDER_LABEL = {1: 'Homme', 2: 'Femme'}

In [ ]:
# Charger les données
records = load_json_file('../CORTI_sample_audiograms_500.json')
df = pd.DataFrame(records)
feature_df, _ = build_feature_matrix(df)
X, scaler, imputer = preprocess(feature_df, fit=True)

# Pipeline ML
scores_df, if_model, ae_model, _ = run_unsupervised_pipeline(
    X, contamination=0.05, ae_epochs=100, feature_df=feature_df
)

# Règles cliniques
nihl_flag = compute_nihl_flag(feature_df)
low_L = feature_df[["L_250", "L_500", "L_1000"]].mean(axis=1)
low_R = feature_df[["R_250", "R_500", "R_1000"]].mean(axis=1)
meniere_flag = ((low_L.fillna(0) > 25) | (low_R.fillna(0) > 25)).astype(int)

scores_df["nihl_flag"] = nihl_flag
scores_df["meniere_flag"] = meniere_flag
scores_df["anomaly_final"] = (
    (scores_df["anomaly_consensus"] == 1) |
    (scores_df["nihl_flag"] == 1) |
    (scores_df["meniere_flag"] == 1)
).astype(int)

print(f"Dataset : {len(df)} records, {df['patient'].nunique()} patients")

## 1. Distribution démographique

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Distribution âge
ages = df['age_at_visit'].dropna()
axes[0].hist(ages, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(ages.mean(), color='red', linestyle='--', label=f'Moyenne: {ages.mean():.1f} ans')
axes[0].axvline(ages.median(), color='orange', linestyle='--', label=f'Médiane: {ages.median():.1f} ans')
axes[0].set_xlabel('Âge (années)')
axes[0].set_ylabel('Nombre de records')
axes[0].set_title(f'Distribution des âges (n={len(ages)})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Distribution genre
gender_counts = df['gender'].value_counts()
gender_labels = [GENDER_LABEL.get(g, f'Genre {g}') for g in gender_counts.index]
axes[1].bar(range(len(gender_counts)), gender_counts.values, 
            color=['steelblue', 'coral'], edgecolor='white', alpha=0.8)
axes[1].set_xticks(range(len(gender_counts)))
axes[1].set_xticklabels(gender_labels)
axes[1].set_ylabel('Nombre de records')
axes[1].set_title('Répartition par genre')
axes[1].grid(True, alpha=0.3, axis='y')
for i, v in enumerate(gender_counts.values):
    axes[1].text(i, v + 5, f'{v}\n({100*v/len(df):.1f}%)', ha='center', fontweight='bold')

# Âge par genre
df_plot = df[df['age_at_visit'].notna()].copy()
df_plot['gender_label'] = df_plot['gender'].map(GENDER_LABEL)
sns.boxplot(data=df_plot, x='gender_label', y='age_at_visit', ax=axes[2], palette=['steelblue', 'coral'])
axes[2].set_xlabel('Genre')
axes[2].set_ylabel('Âge (années)')
axes[2].set_title('Distribution âge par genre')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Distribution PTA (sévérité)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Histogramme PTA gauche
pta_L = feature_df['PTA_L'].dropna()
axes[0, 0].hist(pta_L, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
axes[0, 0].axvline(25, color='red', linestyle='--', linewidth=2, label='Seuil normalité (25 dB)')
axes[0, 0].axvline(pta_L.mean(), color='orange', linestyle='--', label=f'Moyenne: {pta_L.mean():.1f} dB')
axes[0, 0].set_xlabel('PTA Oreille Gauche (dB HL)')
axes[0, 0].set_ylabel('Nombre de records')
axes[0, 0].set_title(f'Distribution PTA OG (n={len(pta_L)})')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Histogramme PTA droite
pta_R = feature_df['PTA_R'].dropna()
axes[0, 1].hist(pta_R, bins=30, color='coral', edgecolor='white', alpha=0.8)
axes[0, 1].axvline(25, color='red', linestyle='--', linewidth=2, label='Seuil normalité (25 dB)')
axes[0, 1].axvline(pta_R.mean(), color='orange', linestyle='--', label=f'Moyenne: {pta_R.mean():.1f} dB')
axes[0, 1].set_xlabel('PTA Oreille Droite (dB HL)')
axes[0, 1].set_ylabel('Nombre de records')
axes[0, 1].set_title(f'Distribution PTA OD (n={len(pta_R)})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# PTA par catégorie de visite
df_pta = pd.DataFrame({
    'PTA_L': feature_df['PTA_L'],
    'PTA_R': feature_df['PTA_R'],
    'visit_category': df['visit_category']
})
df_pta['visit_label'] = df_pta['visit_category'].map(VISIT_CATEGORY_LABELS)
df_pta_long = pd.melt(df_pta, id_vars=['visit_label'], value_vars=['PTA_L', 'PTA_R'], 
                       var_name='Oreille', value_name='PTA')
df_pta_long['Oreille'] = df_pta_long['Oreille'].map({'PTA_L': 'Gauche', 'PTA_R': 'Droite'})

sns.boxplot(data=df_pta_long, x='visit_label', y='PTA', hue='Oreille', ax=axes[1, 0], palette=['steelblue', 'coral'])
axes[1, 0].axhline(25, color='red', linestyle='--', linewidth=1.5, alpha=0.7)
axes[1, 0].set_xlabel('Catégorie de visite')
axes[1, 0].set_ylabel('PTA (dB HL)')
axes[1, 0].set_title('PTA par catégorie de visite')
axes[1, 0].legend(title='Oreille')
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Classification sévérité
def classify_pta(pta):
    if np.isnan(pta):
        return 'Inconnu'
    elif pta < 25:
        return 'Normal (<25 dB)'
    elif pta < 40:
        return 'Léger (25-40 dB)'
    elif pta < 70:
        return 'Modéré (40-70 dB)'
    else:
        return 'Sévère (>70 dB)'

severity_L = feature_df['PTA_L'].apply(classify_pta)
severity_R = feature_df['PTA_R'].apply(classify_pta)
severity_counts = pd.DataFrame({
    'Gauche': severity_L.value_counts(),
    'Droite': severity_R.value_counts()
}).fillna(0)

severity_order = ['Normal (<25 dB)', 'Léger (25-40 dB)', 'Modéré (40-70 dB)', 'Sévère (>70 dB)', 'Inconnu']
severity_counts = severity_counts.reindex(severity_order, fill_value=0)

severity_counts.plot(kind='bar', ax=axes[1, 1], color=['steelblue', 'coral'], 
                     edgecolor='white', alpha=0.8, width=0.7)
axes[1, 1].set_xlabel('Sévérité de la perte')
axes[1, 1].set_ylabel('Nombre de records')
axes[1, 1].set_title('Classification par sévérité (PTA)')
axes[1, 1].legend(title='Oreille')
axes[1, 1].set_xticklabels(severity_order, rotation=45, ha='right')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"\nPTA OG : moyenne={pta_L.mean():.1f} dB, médiane={pta_L.median():.1f} dB")
print(f"PTA OD : moyenne={pta_R.mean():.1f} dB, médiane={pta_R.median():.1f} dB")
print(f"Perte auditive (PTA > 25 dB) : OG={100*(pta_L > 25).mean():.1f}%, OD={100*(pta_R > 25).mean():.1f}%")

## 3. Asymétrie inter-oreilles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution asymétrie
asymmetry = feature_df['asymmetry_mean'].dropna()
axes[0].hist(asymmetry, bins=30, color='purple', edgecolor='white', alpha=0.8)
axes[0].axvline(15, color='red', linestyle='--', linewidth=2, label='Seuil clinique (15 dB)')
axes[0].axvline(asymmetry.mean(), color='orange', linestyle='--', label=f'Moyenne: {asymmetry.mean():.1f} dB')
axes[0].set_xlabel('Asymétrie moyenne |OG - OD| (dB)')
axes[0].set_ylabel('Nombre de records')
axes[0].set_title(f'Distribution asymétrie inter-oreilles (n={len(asymmetry)})')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Scatter PTA_L vs PTA_R
df_scatter = pd.DataFrame({
    'PTA_L': feature_df['PTA_L'],
    'PTA_R': feature_df['PTA_R'],
    'asymmetry': feature_df['asymmetry_mean']
}).dropna()

scatter = axes[1].scatter(df_scatter['PTA_L'], df_scatter['PTA_R'], 
                          c=df_scatter['asymmetry'], cmap='YlOrRd', 
                          s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1].plot([0, 120], [0, 120], 'k--', linewidth=1.5, alpha=0.5, label='OG = OD')
axes[1].plot([0, 120], [15, 135], 'r--', linewidth=1, alpha=0.5, label='Asymétrie 15 dB')
axes[1].plot([15, 135], [0, 120], 'r--', linewidth=1, alpha=0.5)
axes[1].set_xlabel('PTA Oreille Gauche (dB HL)')
axes[1].set_ylabel('PTA Oreille Droite (dB HL)')
axes[1].set_title('Corrélation PTA OG vs OD (couleur = asymétrie)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_xlim(-5, max(df_scatter['PTA_L'].max(), df_scatter['PTA_R'].max()) + 5)
axes[1].set_ylim(-5, max(df_scatter['PTA_L'].max(), df_scatter['PTA_R'].max()) + 5)

cbar = plt.colorbar(scatter, ax=axes[1])
cbar.set_label('Asymétrie (dB)', rotation=270, labelpad=20)

plt.tight_layout()
plt.show()

print(f"\nAsymétrie moyenne : {asymmetry.mean():.1f} dB")
print(f"Asymétrie > 15 dB : {100*(asymmetry > 15).mean():.1f}%")

## 4. Audiogrammes moyens par catégorie

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for visit_cat, label in VISIT_CATEGORY_LABELS.items():
    mask = df['visit_category'] == visit_cat
    if mask.sum() == 0:
        continue
    
    means_L = [feature_df.loc[mask, f'L_{f}'].mean() for f in STANDARD_FREQS]
    axes[0].plot(STANDARD_FREQS, means_L, marker='x', linewidth=2, label=f'{label} (n={mask.sum()})')
    
    means_R = [feature_df.loc[mask, f'R_{f}'].mean() for f in STANDARD_FREQS]
    axes[1].plot(STANDARD_FREQS, means_R, marker='o', linewidth=2, label=f'{label} (n={mask.sum()})')

for ax, title in zip(axes, ['Oreille Gauche', 'Oreille Droite']):
    ax.axhline(25, color='red', linestyle='--', linewidth=1.5, alpha=0.5, label='Seuil normalité')
    ax.set_xlabel('Fréquence (Hz)')
    ax.set_ylabel('Seuil auditif moyen (dB HL)')
    ax.set_title(f'Audiogramme moyen — {title}')
    ax.set_xscale('log')
    ax.set_xticks(STANDARD_FREQS)
    ax.set_xticklabels([f'{f}' for f in STANDARD_FREQS])
    ax.invert_yaxis()
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Corrélation scores ML vs features cliniques

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap corrélation
clinical_features = [
    'PTA_L', 'PTA_R', 'asymmetry_mean', 
    'high_freq_drop_L', 'high_freq_drop_R',
    'notch_4k_L', 'notch_4k_R',
    'low_freq_pta_L', 'low_freq_pta_R'
]

corr_df = pd.concat([
    feature_df[clinical_features],
    scores_df[['anomaly_score_if', 'reconstruction_error', 'pca_reconstruction_error']]
], axis=1)

corr_matrix = corr_df.corr().loc[clinical_features, ['anomaly_score_if', 'reconstruction_error', 'pca_reconstruction_error']]

sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r', center=0, 
            vmin=-1, vmax=1, ax=axes[0], cbar_kws={'label': 'Corrélation'})
axes[0].set_title('Corrélation features cliniques vs scores ML')
axes[0].set_xlabel('Scores ML')
axes[0].set_ylabel('Features cliniques')

# Scatter reconstruction error vs PTA
pta_mean = (feature_df['PTA_L'] + feature_df['PTA_R']) / 2
scatter = axes[1].scatter(pta_mean, scores_df['reconstruction_error'], 
                          c=scores_df['anomaly_consensus'], cmap='RdYlGn_r',
                          s=50, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel('PTA moyen (OG + OD) / 2 (dB HL)')
axes[1].set_ylabel('Erreur de reconstruction (Autoencoder)')
axes[1].set_title('Relation PTA vs Score ML')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[1], ticks=[0, 1])
cbar.set_label('Consensus', rotation=270, labelpad=20)
cbar.ax.set_yticklabels(['Normal', 'Anomalie'])

plt.tight_layout()
plt.show()

## 6. Trajectoires patients (Baseline → Periodic)

In [ ]:
# Patients avec trajectoire
patient_visits = df.groupby('patient')['visit_category'].apply(list)
patients_with_trajectory = patient_visits[
    patient_visits.apply(lambda x: 0 in x and (1 in x or 2 in x))
].index.tolist()

print(f"Patients avec trajectoire : {len(patients_with_trajectory)} / {df['patient'].nunique()}")

if len(patients_with_trajectory) > 0:
    for i, patient in enumerate(patients_with_trajectory[:3], 1):
        patient_df = df[df['patient'] == patient].sort_values('visit_date')
        print(f"\nPatient {i} : {len(patient_df)} visites")
        
        fig = plot_patient_trajectory(df, patient)
        plt.show()

## 7. Matrice confusion ML vs Clinique

In [ ]:
# ML vs règles cliniques
ml_flags = scores_df['anomaly_consensus'].values
clinical_flags = (
    (scores_df['nihl_flag'] == 1) | 
    (scores_df['meniere_flag'] == 1)
).astype(int).values

cm = confusion_matrix(clinical_flags, ml_flags)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, 
                               display_labels=['Normal', 'Anomalie'])

fig, ax = plt.subplots(figsize=(8, 6))
disp.plot(ax=ax, cmap='Blues', values_format='d')
ax.set_xlabel('Prédiction ML (consensus)')
ax.set_ylabel('Vérité terrain (flags cliniques)')
ax.set_title('Matrice de confusion : ML vs Règles cliniques')
plt.tight_layout()
plt.show()

# Métriques
from sklearn.metrics import precision_score, recall_score, f1_score

if clinical_flags.sum() > 0:
    precision = precision_score(clinical_flags, ml_flags, zero_division=0)
    recall = recall_score(clinical_flags, ml_flags, zero_division=0)
    f1 = f1_score(clinical_flags, ml_flags, zero_division=0)
    
    print(f"\nPrecision : {precision:.2%}")
    print(f"Recall    : {recall:.2%}")
    print(f"F1-score  : {f1:.2%}")